In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

Matplotlib is building the font cache; this may take a moment.


In [4]:
# read in all words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [5]:
len(words)

32033

In [6]:
# build the vocabulary of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [128]:
# build the dataset

block_size = 3 # context length: how many characters do we take to predict the next one?
X_list, Y_list = [], []
for w in words[:]:
    
    # print(w)
    context = [0] * block_size # initialize with all start characters
    for ch in w + '.': # for each character plus the stop character
        ix = stoi[ch] # index of the character
        X_list.append(context) # add the current context to the inputs
        Y_list.append(ix) # add the next character to the targets
        # print(''.join(itos[i] for i in context), '----->', ch, '[ math representation: ',','.join(str(i) for i in context), '----->', ix, ']')
        context = context[1:] + [ix] # slide the context window, append the new character
        
X = torch.tensor(X_list)
Y = torch.tensor(Y_list)

In [165]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [166]:
# build dataset splits

def build_dataset(words):
    block_size = 3 # context length: how many characters do we take to predict the next one?
    X_list, Y_list = [], []
    for w in words:
        
        context = [0] * block_size # initialize with all start characters
        for ch in w + '.': # for each character plus the stop character
            ix = stoi[ch] # index of the character
            X_list.append(context) # add the current context to the inputs
            Y_list.append(ix) # add the next character to the targets
            context = context[1:] + [ix] # slide the context window, append the new character
            
    X = torch.tensor(X_list)
    Y = torch.tensor(Y_list)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])
  
 

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


In [ ]:
C = torch.randn((27, 2)) # 27 characters, 2-dimensional character embeddings, the lookup table 
C

In [131]:
F.one_hot(torch.tensor(5), num_classes=27).float() @ C

tensor([-0.9550, -1.3980])

In [ ]:
emb = C[X] 
print(emb.shape)
emb

In [133]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [134]:
torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], dim=1).shape
# why we cat in this way?

torch.Size([228146, 6])

In [135]:
torch.cat(torch.unbind(emb, dim=1), dim=1).shape

torch.Size([228146, 6])

In [136]:
h = torch.tanh(emb.view(emb.shape[0], -1) @ W1 + b1) # hidden layer
h

tensor([[-0.9144, -0.9923, -0.7923,  ...,  0.2479, -0.7990, -0.7951],
        [-0.9831, -0.8073, -0.9997,  ...,  0.9973,  0.7102,  0.9985],
        [-0.8009, -0.9170,  0.9299,  ...,  0.0017,  0.9857,  0.9953],
        ...,
        [ 0.7952, -0.9257, -0.6161,  ...,  0.9887, -0.8699,  0.9881],
        [ 0.6994, -0.8985,  0.9915,  ...,  0.9483, -0.8464,  0.8849],
        [ 0.9992, -0.9836,  0.4719,  ...,  0.9294, -0.9998,  0.2611]])

In [137]:
W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [138]:
logits = h @ W2 + b2
logits.shape

torch.Size([228146, 27])

In [139]:
counts = logits.exp()
probs = counts / counts.sum(1, keepdim=True)
probs.shape

torch.Size([228146, 27])

In [140]:
 X.shape, Y.shape

(torch.Size([228146, 3]), torch.Size([228146]))

In [177]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g) # 27 characters, 2-dimensional character embeddings, the lookup table
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [178]:
sum(p.nelement() for p in parameters)

3481

In [179]:
for p in parameters:
    p.requires_grad = True

In [180]:
lre = torch.linspace(-3, 0, 1000)
lrs = 10**lre

In [ ]:
for _ in range(10000):
    # minibtach construction
    minibatch_size = 32
    batch_indices = torch.randint(0, Xtr.shape[0], (minibatch_size,))
    
    # forward pass
    emb = C[Xtr[batch_indices]]
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[batch_indices])
    # print(loss.item())
    # backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # update
    for p in parameters:
        p.data += -0.1 * p.grad # how do you determine the learning rate?
        
    # track stats
    
        

In [183]:
# trainig split, validation split, test split
# 80% training, 10% validation, 10% test
emb = C[Xtr]
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ytr)
loss.item()

2.464359760284424

In [184]:
# trainig split, validation split, test split
# 80% training, 10% validation, 10% test
emb = C[Xdev]
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)
logits = h @ W2 + b2
loss = F.cross_entropy(logits, Ydev)
loss.item()

2.4539475440979004